# Roofline Analysis Framework — Demo Notebook

This notebook walks through **layer-wise roofline analysis** for three model examples:

1. **Custom MLP** — simple, no external downloads needed
2. **ResNet-50** — CNN with Conv2d / BatchNorm / Linear layers
3. **BERT-base** — Transformer with attention and normalization layers

Plus:
- Using the **folder** and **zip** input modes
- Running analysis in **training mode**
- Comparing the same model across **multiple hardware targets**
- **Fetching specs** for hardware not in the built-in database


In [ ]:
# Install the framework (run once)
# !pip install -e .. --quiet

import sys, os
sys.path.insert(0, os.path.abspath('..'))

from roofline import analyze, HW_DB
from roofline.hardware.hw_database import HW_DB

print('Available hardware targets:', list(HW_DB.keys()))

---
## 1. Custom MLP — No Downloads Required

A simple 4-layer MLP. We pass a `torch.nn.Module` directly.

In [ ]:
import torch
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self, in_dim=784, hidden=2048, out_dim=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.GELU(),
            nn.LayerNorm(hidden),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.LayerNorm(hidden),
            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Linear(hidden // 2, out_dim),
        )

    def forward(self, x):
        return self.net(x)

mlp = MLP()

results_mlp = analyze(
    model=mlp,
    input_shapes=[(1, 784)],
    hw=HW_DB['h100_sxm'],
    dtype='float16',
    mode='inference',
)

results_mlp.print_table()

In [ ]:
# Roofline plot
results_mlp.plot_roofline(figsize=(12, 6));

In [ ]:
# Export to DataFrame for custom analysis
df = results_mlp.to_dataframe()
df[['name', 'type', 'flops', 'total_bytes', 'arithmetic_intensity', 'bottleneck', 'theoretical_time_ms']].head(20)

---
## 2. ResNet-50 — CNN Analysis

In [ ]:
import torchvision.models as models

resnet = models.resnet50(weights=None)

results_resnet = analyze(
    model=resnet,
    input_shapes=[(1, 3, 224, 224)],
    hw=HW_DB['a100_80gb'],
    dtype='float16',
    mode='inference',
)

print(results_resnet.summary())

In [ ]:
# Top 10 layers by FLOPs
results_resnet.print_table(top_n=10, sort_by='flops')

In [ ]:
results_resnet.plot_roofline(annotate_top_n=5);

In [ ]:
# Batch-size sensitivity: compare batch=1 vs batch=32 vs batch=128
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, batch in zip(axes, [1, 32, 128]):
    r = analyze(
        model=resnet,
        input_shapes=[(batch, 3, 224, 224)],
        hw=HW_DB['a100_80gb'],
        dtype='float16',
    )
    from roofline.reporting.roofline_plot import RooflinePlot
    RooflinePlot(r).plot(figsize=(6, 5), show=False)
    ax.set_title(f'Batch={batch}')
plt.suptitle('ResNet-50 — Roofline at Different Batch Sizes', y=1.02, fontsize=14)
plt.tight_layout()
plt.show();

---
## 3. BERT-base — Transformer Analysis

> Requires: `pip install transformers`

In [ ]:
results_bert = analyze(
    model='bert-base-uncased',
    source='huggingface',
    input_shapes=[(1, 128)],    # batch=1, seq_len=128
    hw=HW_DB['h100_sxm'],
    dtype='float16',
    mode='inference',
)

print(results_bert.summary())

In [ ]:
results_bert.print_table(top_n=15, sort_by='flops')

In [ ]:
results_bert.plot_roofline(annotate_top_n=6);

---
## 4. Training Mode Analysis

Training mode adds gradient bytes (≈ weight bytes) and Adam optimizer state bytes (2 × params × 4 bytes).

In [ ]:
results_train = analyze(
    model=resnet,
    input_shapes=[(32, 3, 224, 224)],
    hw=HW_DB['h100_sxm'],
    dtype='float32',
    mode='training',
)

results_infer = analyze(
    model=resnet,
    input_shapes=[(32, 3, 224, 224)],
    hw=HW_DB['h100_sxm'],
    dtype='float32',
    mode='inference',
)

print('=== INFERENCE ===')
print(f'  Total memory : {results_infer.total_bytes/1e9:.2f} GB')
print(f'  Est. time    : {results_infer.theoretical_time_ms:.2f} ms')
print()
print('=== TRAINING ===')
print(f'  Total memory : {results_train.total_bytes/1e9:.2f} GB')
print(f'  Est. time    : {results_train.theoretical_time_ms:.2f} ms')

---
## 5. Multi-Hardware Comparison

Compare the same model across H100, A100, RTX 4090, and Apple M4 Max.

In [ ]:
import pandas as pd

hw_targets = ['h100_sxm', 'a100_80gb', 'rtx_4090', 'm4_max']
rows = []

for hw_key in hw_targets:
    r = analyze(
        model=mlp,
        input_shapes=[(1, 784)],
        hw=HW_DB[hw_key],
        dtype='float16',
    )
    rows.append({
        'Hardware': HW_DB[hw_key].name,
        'FP16 TFLOPs': HW_DB[hw_key].peak_flops.get('float16', 0) / 1e12,
        'Mem BW (TB/s)': HW_DB[hw_key].peak_mem_bw / 1e12,
        'Est. Time (ms)': r.theoretical_time_ms,
        'Memory-bound layers': len(r.memory_bound_layers()),
        'Compute-bound layers': len(r.compute_bound_layers()),
    })

pd.DataFrame(rows).set_index('Hardware')

---
## 6. Folder Input — Manual Layer Files

If you have individual layer files saved as `.pt` files in a folder:

In [ ]:
import os, torch

# Create a demo folder with layer files
os.makedirs('/tmp/demo_layers', exist_ok=True)

class EmbedBlock(torch.nn.Module):
    def __init__(self): super().__init__(); self.embed = torch.nn.Embedding(50000, 768)
    def forward(self, x): return self.embed(x)

class AttentionBlock(torch.nn.Module):
    def __init__(self): super().__init__(); self.attn = torch.nn.MultiheadAttention(768, 12, batch_first=True)
    def forward(self, x): return self.attn(x, x, x)

class FFNBlock(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.net = torch.nn.Sequential(torch.nn.Linear(768, 3072), torch.nn.GELU(), torch.nn.Linear(3072, 768))
    def forward(self, x): return self.net(x)

torch.save(EmbedBlock(), '/tmp/demo_layers/00_embed.pt')
torch.save(AttentionBlock(), '/tmp/demo_layers/01_attention.pt')
torch.save(FFNBlock(), '/tmp/demo_layers/02_ffn.pt')

print('Saved layer files:', os.listdir('/tmp/demo_layers'))

In [ ]:
results_folder = analyze(
    model='/tmp/demo_layers/',
    input_shapes=[(1, 128)],
    hw=HW_DB['h100_sxm'],
    dtype='float16',
)

results_folder.print_table()

---
## 7. Zip Input — Drag & Drop Upload

Use the file upload widget to upload a `.zip` archive directly.

In [ ]:
# Option A: ipywidgets drag-and-drop upload
try:
    import ipywidgets as widgets
    from IPython.display import display
    import io

    upload = widgets.FileUpload(accept='.zip', multiple=False, description='Upload .zip model')
    display(upload)
    print('Upload a .zip file above, then run the next cell.')
except ImportError:
    print('ipywidgets not installed. Use Option B below instead.')

In [ ]:
# Option A continued: analyze after upload
try:
    if upload.value:
        uploaded_file = list(upload.value.values())[0]
        zip_bytes = uploaded_file['content']
        zip_path = '/tmp/uploaded_model.zip'
        with open(zip_path, 'wb') as f:
            f.write(zip_bytes)

        results_zip = analyze(
            model=zip_path,
            input_shapes=[(1, 3, 224, 224)],   # ← adjust to your model
            hw=HW_DB['h100_sxm'],
            dtype='float16',
        )
        results_zip.print_table()
    else:
        print('No file uploaded yet.')
except NameError:
    pass

In [ ]:
# Option B: provide zip path directly
# results_zip = analyze(
#     model='./my_model.zip',
#     input_shapes=[(1, 3, 224, 224)],
#     hw=HW_DB['h100_sxm'],
#     dtype='float16',
# )
# results_zip.print_table()

---
## 8. Auto-Detect Local Hardware

In [ ]:
from roofline import detect_hw

try:
    local_hw = detect_hw()
    print(f'Detected: {local_hw}')

    results_local = analyze(
        model=mlp,
        input_shapes=[(1, 784)],
        hw=local_hw,
        dtype='float16',
    )
    results_local.print_table(top_n=5)
except Exception as e:
    print(f'Detection failed or not in DB: {e}')

---
## 9. Fetch Hardware Specs from the Web

If the hardware you want is not in the built-in database, fetch it from TechPowerUp:

In [ ]:
from roofline import fetch_hw

# Uncomment and set fetch_from_web=True to allow network access
# hw_custom = fetch_hw('RTX 5090', fetch_from_web=True)
# print(hw_custom)

# results_custom = analyze(
#     model=mlp,
#     input_shapes=[(1, 784)],
#     hw=hw_custom,
#     dtype='float16',
# )
# results_custom.print_table()

print('Uncomment lines above and set fetch_from_web=True to fetch custom HW specs.')

---
## 10. Custom FLOPs Handler

Register a handler for a layer type not covered by default:

In [ ]:
from roofline.core.flop_counter import FlopCounter
from roofline.core.analyzer import RooflineAnalyzer

# Create analyzer and register a custom handler
analyzer = RooflineAnalyzer()

@analyzer.flop_counter.register('MyFusedOp')
def count_my_fused_op(layer):
    # Example: fused multiply-add + activation, ~3 ops per element
    n = layer.attrs.get('n_elements', 0)
    return 3 * n

print('Custom handler registered for MyFusedOp')